# Trabajo Práctico Final: Visualización de Datos
## Visualización de Datos utilizando la World Bank Data360

**Curso:** Visualización de la Información  
**Tema:** Análisis de Inversión y Resultados Educativos mediante World Bank Data360 API

---

### Objetivos del Proyecto
Explorar la API de datos del Banco Mundial para identificar bases de datos e indicadores disponibles.

* **Exploración de Bases de Datos:** Identificación de las bases de datos existentes en la plataforma.
* **Identificación de Indicadores:** Relevamiento de indicadores disponibles para el análisis.
* **Estructura de Datos:** Análisis de la organización de la información (países, series temporales, temáticas, etc.).

---

### Integrantes
* **Schelp**, Martin
* **Cozzi**, Alejandro Luis
* **Blanco Villar**, Maximiliano Jesús

In [68]:
%pip install -q "vegafusion[embed]>=1.5.0" "vegafusion>=1.5.0" "vl-convert-python>=1.6.0"
%pip install pandas
%pip install altair
%pip install duckdb
%pip install requests
%pip install --upgrade certifi
%pip install --upgrade requests urllib3
%pip install python-certifi-win32
%pip install python-dotenv
%pip install pyarrow
%pip install vega_datasets

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [69]:
import pandas as pd
import altair as alt
import duckdb
import requests
import certifi
import json
import os

In [70]:
alt.data_transformers.enable('vegafusion')
alt.renderers.enable('colab')

RendererRegistry.enable('colab')

In [71]:
response = requests.get("https://data360api.worldbank.org/data360/indicators?datasetId=WB_WDI")
response.status_code

200

# Identificar bases

In [ ]:
url = "https://data360api.worldbank.org/data360/searchv2"
headers = {"accept": "*/*", "Content-Type": "application/json"}

all_results = []
skip = 0
top = 1000

while True:
    payload = {
        "count": True,
        "select": "series_description/database_id, series_description/database_name",
        "top": top,
        "skip": skip
    }
    r = requests.post(url, json=payload, headers=headers).json()
    results = r.get("value", [])
    if not results:
        break
    all_results.extend(results)
    skip += top
    total = r.get("@odata.count", 0)
    print(f"  Procesados: {skip}/{total}")
    if skip >= total:
        break

df_dbs = (
    pd.json_normalize(all_results)[['series_description.database_id', 'series_description.database_name']]
    .rename(columns={
        'series_description.database_id':   'database_id',
        'series_description.database_name': 'database_name'
    })
    .dropna(subset=['database_id'])
    .drop_duplicates()
    .sort_values('database_id')
    .reset_index(drop=True)
)



  Procesados: 1000/12938
  Procesados: 2000/12938
  Procesados: 3000/12938
  Procesados: 4000/12938
  Procesados: 5000/12938
  Procesados: 6000/12938
  Procesados: 7000/12938
  Procesados: 8000/12938
  Procesados: 9000/12938
  Procesados: 10000/12938
  Procesados: 11000/12938
  Procesados: 12000/12938
  Procesados: 13000/12938

Total de bases de datos: 161
                    database_id                                                                                                        database_name
0                        BS_BTI                                                                The Bertelsmann Stiftung’s Transformation Index (BTI)
1                        BS_SGI                                                                              Sustainable Governance Indicators (SGI)
2                  DRMKC_INFORM                                                                                                    INFORM Risk Index
3                     EIA_EIAOD              

In [ ]:
print(f"\nLa api da acceso a un total de {df_dbs['database_id'].nunique()} bases de datos. ")
print(df_dbs.to_string())


Total de bases de datos: 161
                    database_id                                                                                                        database_name
0                        BS_BTI                                                                The Bertelsmann Stiftung’s Transformation Index (BTI)
1                        BS_SGI                                                                              Sustainable Governance Indicators (SGI)
2                  DRMKC_INFORM                                                                                                    INFORM Risk Index
3                     EIA_EIAOD                                                                               U.S. Energy Information Administration
4                        EIU_DI                                                                                                      Democracy Index
5                        EPC_AI                                             

In [72]:
def get_wb_data(indicator_id):
    db = "_".join(indicator_id.split("_")[:2])
    base_url = "https://data360api.worldbank.org/data360/data"
    headers = {"accept": "*/*"}

    all_results = []
    skip = 0
    top = 1000

    r = requests.get(
        base_url,
        params={"DATABASE_ID": db, "INDICATOR": indicator_id},
        headers=headers
    ).json()

    total = r.get("count", 0)
    all_results.extend(r.get("value", []))
    print(f"  {indicator_id}: total={total}")

    if len(all_results) >= total:
        pass
    else:
        skip += top
        while skip < total:
            r = requests.get(
                base_url,
                params={"DATABASE_ID": db, "INDICATOR": indicator_id, "skip": skip, "top": top},
                headers=headers
            ).json()
            chunk = r.get("value", [])
            if not chunk:
                break
            all_results.extend(chunk)
            skip += top
            print(f"  {indicator_id}: {min(skip, total)}/{total}")

    if not all_results:
        return pd.DataFrame()

    return pd.json_normalize(all_results).rename(columns={
        "REF_AREA":    "iso3",
        "TIME_PERIOD": "year",
        "OBS_VALUE":   "val",
    })

In [73]:
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()

try:
    import google.colab
    from google.colab import drive, userdata

    drive.mount('/content/drive')

    DATA_DIR = Path(
        userdata.get('DATA_DIR')
    )

except Exception:
    DATA_DIR = Path(
        os.getenv("DATA_DIR", "./data_cache")
    )

DATA_DIR.mkdir(parents=True, exist_ok=True)

print(DATA_DIR)

C:\Users\u632113\itba_world_bank_grupo2\data_cache


In [74]:
INDICATORS = {
    "gasto_edu": "WB_WDI_SE_XPD_TOTL_GD_ZS",
    "pib_pc":    "WB_WDI_NY_GDP_PCAP_CD",
    "prim":      "WB_GS_SE_PRM_ENRR",
    "sec":       "WB_GS_SE_SEC_ENRR",
    "ter":       "WB_GS_SE_TER_ENRR",
    "comp":      "WB_GS_SE_PRM_CMPT_ZS",
}

def load_or_fetch(name, indicator_id):
    path = os.path.join(DATA_DIR, f"{name}.parquet")
    if os.path.exists(path):
        print(f"  {name}: cargando desde disco")
        return pd.read_parquet(path)
    print(f"  {name}: descargando...")
    df = get_wb_data(indicator_id)
    df.to_parquet(path, index=False)
    return df

gasto_edu, pib_pc, prim, sec, ter, comp = [
    load_or_fetch(name, ind) for name, ind in INDICATORS.items()
]

  gasto_edu: cargando desde disco
  pib_pc: cargando desde disco
  prim: cargando desde disco
  sec: cargando desde disco
  ter: cargando desde disco
  comp: cargando desde disco


In [75]:
print(gasto_edu.head())
type(gasto_edu)
len(gasto_edu)

       val TIME_FORMAT  UNIT_MULT COMMENT_OBS OBS_STATUS OBS_CONF AGG_METHOD  \
0  6.16265         P1Y          0        None          A       PU         _Z   
1  1.77495         P1Y          0        None          A       PU         _Z   
2  3.60613         P1Y          0        None          A       PU         _Z   
3  2.46335         P1Y          0        None          A       PU         _Z   
4  2.75565         P1Y          0        None          A       PU         _Z   

  DECIMALS COMMENT_TS DATA_SOURCE  ...  SEX AGE URBANISATION COMP_BREAKDOWN_1  \
0        2       None        None  ...   _T  _T           _T               _Z   
1        2       None        None  ...   _T  _T           _T               _Z   
2        2       None        None  ...   _T  _T           _T               _Z   
3        2       None        None  ...   _T  _T           _T               _Z   
4        2       None        None  ...   _T  _T           _T               _Z   

  COMP_BREAKDOWN_2 COMP_BREAKDOW

6341

In [76]:
def get_regions():
    path = os.path.join(DATA_DIR, "reg_df.parquet")
    if os.path.exists(path):
        print("  reg_df: cargando desde Drive")
        return pd.read_parquet(path)
    print("  reg_df: descargando desde API WB...")
    reg_raw = requests.get(
        "https://api.worldbank.org/v2/country?format=json&per_page=1000"
    ).json()
    reg_df = pd.json_normalize(reg_raw[1])[['id', 'region.value']].rename(
        columns={'id': 'iso3', 'region.value': 'region'}
    )
    reg_df = reg_df[reg_df['region'] != 'Aggregates']
    reg_df.to_parquet(path, index=False)
    print("  reg_df: guardado")
    return reg_df

reg_df = get_regions()

  reg_df: cargando desde Drive


In [77]:
df_viz1 = gasto_edu.merge(pib_pc, on=['iso3', 'year']).merge(reg_df, on='iso3')
df_viz1['year'] = pd.to_numeric(df_viz1['year'])
df_viz1 = df_viz1.dropna(subset=['val_x', 'val_y'])
df_viz1 = df_viz1.rename(columns={'country.value_x': 'country_name'})
slider = alt.binding_range(min=2000, max=2022, step=1, name='Año: ')
select_year = alt.selection_point(fields=['year'], bind=slider, value=2019, name="Year")

chart = alt.Chart(df_viz1).mark_circle(size=100, opacity=0.7).encode(
    x=alt.X("val_y:Q", title="PIB per cápita (USD)", scale=alt.Scale(type='log'), axis=alt.Axis(grid=False)),
    y=alt.Y("val_x:Q", title="Gasto en Educación (% PIB)", scale=alt.Scale(domain=[0, 15])),
    color=alt.Color("region:N", title="Región"),
    tooltip=[
        alt.Tooltip("iso3:N", title="País"),
        alt.Tooltip("year:O", title="Año"),
        alt.Tooltip("val_y:Q", title="PIB p/c", format="$,.0f"),
        alt.Tooltip("val_x:Q", title="% Gasto", format=".1f")
    ]
).add_params(select_year).transform_filter(select_year).properties(
    width=600, height=400, title="Inversión Educativa vs Riqueza (Escala Logarítmica)"
).interactive()

chart

alt.Chart(...)

In [78]:
import pandas as pd
import altair as alt

# 1. Concatenación
df_traj = pd.concat([
    prim[['iso3', 'year', 'val', 'SEX', 'AGE', 'URBANISATION', 'COMP_BREAKDOWN_1']].assign(nivel="Primaria"),
    sec[['iso3', 'year', 'val', 'SEX', 'AGE', 'URBANISATION', 'COMP_BREAKDOWN_1']].assign(nivel="Secundaria"),
    ter[['iso3', 'year', 'val', 'SEX', 'AGE', 'URBANISATION', 'COMP_BREAKDOWN_1']].assign(nivel="Terciaria")
])

# 2. Filtrado CORREGIDO (Aquí está la magia)
df_traj = df_traj[
    (df_traj['SEX'] == '_T') &
    (df_traj['AGE'] == '_T') &
    (df_traj['URBANISATION'] == '_T') &
    (df_traj['COMP_BREAKDOWN_1'].isin(['GS_DXT', '_Z'])) # <--- Ahora acepta Terciaria
]

# 3. Creación del subset para Argentina
df_arg_traj = (
    df_traj[df_traj['iso3'] == 'ARG']
    .dropna(subset=['val'])
    [['iso3', 'year', 'val', 'nivel']]
    .copy()
)

# Convertir a numérico
df_arg_traj['year'] = pd.to_numeric(df_arg_traj['year'])
df_arg_traj['val']  = pd.to_numeric(df_arg_traj['val'])

In [79]:
# =========================
# Gobiernos y partidos
# =========================

gobiernos = pd.DataFrame({
    'year_start': [
        1970, 1971, 1973, 1973, 1974,
        1976, 1983, 1989, 1999, 2001,
        2002, 2003, 2007, 2015, 2019
    ],
    'year_end': [
        1971, 1973, 1973, 1974, 1976,
        1983, 1989, 1999, 2001, 2002,
        2003, 2007, 2015, 2019, 2022
    ],
    'presidente': [
        'Levingston', 'Lanusse', 'Cámpora', 'Perón', 'Isabel Perón',
        'Dictadura Militar', 'Alfonsín', 'Menem', 'De la Rúa',
        'Rodríguez Saá', 'Duhalde', 'Kirchner', 'CFK',
        'Macri', 'Alberto Fernández'
    ],
    'partido': [
        'De Facto', 'De Facto', 'Peronismo', 'Peronismo', 'Peronismo',
        'De Facto', 'UCR', 'Peronismo', 'UCR',
        'Peronismo', 'Peronismo', 'Peronismo', 'Peronismo',
        'PRO', 'Peronismo'
    ]
})

# =========================
# Hitos educativos
# =========================

hitos_educativos = pd.DataFrame({
    'year': [1993, 2006],
    'hito': [
        'Ley Federal\nde Educación',
        'Ley de Educación\nNacional'
    ]
})

# =========================
# Selecciones interactivas
# =========================

seleccion_nivel = alt.selection_point(
    fields=['nivel'],
    bind='legend'
)

seleccion_partido = alt.selection_point(
    fields=['partido'],
    bind='legend'
)

# =========================
# Fondos por gobierno
# =========================

rangos = (
    alt.Chart(gobiernos)
    .mark_rect(
        clip=True
    )
    .encode(
        x=alt.X("year_start:Q"),
        x2="year_end:Q",
        color=alt.Color(
            "partido:N",
            scale=alt.Scale(
                domain=['Peronismo', 'UCR', 'PRO', 'De Facto'],
                range=['#2E74C0', '#C62828', '#F9A825', '#616161']
            ),
            legend=alt.Legend(title="Espacio político")
        ),
        opacity=alt.condition(
            seleccion_partido,
            alt.value(0.18),
            alt.value(0.04)
        ),
        tooltip=[
            alt.Tooltip("presidente:N", title="Gobierno"),
            alt.Tooltip("partido:N", title="Partido")
        ]
    )
    .add_params(seleccion_partido)
)

# =========================
# Reglas verticales
# =========================

reglas = (
    alt.Chart(gobiernos)
    .mark_rule(
        strokeDash=[6, 4],
        strokeWidth=1,
        color='gray',
        opacity=0.25,
        clip=True
    )
    .encode(
        x=alt.X("year_start:Q")
    )
)

# =========================
# Hitos educativos
# =========================

lineas_hitos = (
    alt.Chart(hitos_educativos)
    .mark_rule(
        color='black',
        strokeWidth=2,
        strokeDash=[4, 4],
        opacity=0.8,
        clip=True
    )
    .encode(
        x=alt.X("year:Q"),
        tooltip=[
            alt.Tooltip("year:Q", title="Año"),
            alt.Tooltip("hito:N", title="Hito")
        ]
    )
)

texto_hitos = (
    alt.Chart(hitos_educativos)
    .mark_text(
        angle=0,
        align='left',
        baseline='bottom',
        lineBreak='\n',
        dx=5,
        dy=-5,
        color='blue',
        fontWeight='bold',
        fontSize=11,
        clip=True
    )
    .encode(
        x=alt.X("year:Q"),
        y=alt.value(390),
        text=alt.Text("hito:N")
    )
)

# =========================
# Trayectorias
# =========================

chart_traj = (
    alt.Chart(df_arg_traj)
    .mark_line(point=True)
    .encode(
        x=alt.X(
            "year:Q",
            title="Año",
            axis=alt.Axis(format='d', tickCount=10, grid=False),
            scale=alt.Scale(zero=False, domain=[1970, 2022])
        ),
        y=alt.Y(
            "val:Q",
            title="Tasa de matrícula (% bruta)",
            scale=alt.Scale(zero=False)
        ),
        color=alt.Color(
            "nivel:O",
            title="Nivel Educativo",
            sort=['Primaria', 'Secundaria', 'Terciaria']
        ),
        opacity=alt.condition(
            seleccion_nivel,
            alt.value(1),
            alt.value(0.15)
        ),
        size=alt.condition(
            seleccion_nivel,
            alt.value(3),
            alt.value(1.5)
        ),
        tooltip=[
            alt.Tooltip("year:Q", title="Año"),
            alt.Tooltip("nivel:N", title="Nivel"),
            alt.Tooltip("val:Q", title="Tasa (%)", format=".2f")
        ]
    )
    .add_params(seleccion_nivel)
)

# =========================
# Gráfico final
# =========================

final_chart = (
    rangos
    + reglas
    + lineas_hitos
    + texto_hitos
    + chart_traj
).resolve_scale(
    x='shared',
    color='independent'
).properties(
    width=700,
    height=400,
    title="Evolución de la Matrícula por Nivel Educativo en Argentina"
)

final_chart.display()

alt.LayerChart(...)

In [80]:
from vega_datasets import data
iso_map = pd.read_csv("https://raw.githubusercontent.com/lukes/ISO-3166-Countries-with-Regional-Codes/master/all/all.csv")[["alpha-3", "country-code"]].rename(columns={"alpha-3": "iso3", "country-code": "id_num"})
df_map = comp[comp['year'] == '2020'].merge(iso_map, on='iso3')

world = alt.topo_feature(data.world_110m.url, 'countries')

mapa_base = alt.Chart(world).mark_geoshape(
    fill='lightgray',
    stroke='white',
    strokeWidth=0.5
).project('naturalEarth1')

mapa_datos = alt.Chart(world).mark_geoshape().encode(
    color=alt.Color(
        'val:Q',
        title='% Finalización',
        scale=alt.Scale(scheme='viridis', domain=[50, 100])
    ),
    tooltip=[
        alt.Tooltip('iso3:N', title='Código'),
        alt.Tooltip('val:Q', title='% Tasa', format='.2f')
    ]
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(df_map, 'id_num', ['val', 'iso3'])
).project('naturalEarth1')

mapa_final = (mapa_base + mapa_datos).properties(
    width=700,
    height=400,
    title="Acceso Universal: Tasa de Finalización Primaria (2020)"
)

mapa_final.display()

alt.LayerChart(...)